In [133]:
# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0"
os.environ["MUJOCO_GL"] = "glfw"

# datasets
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi

# robots
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.cameras.configs import Cv2Rotation
from lerobot.teleoperators.so101_leader import SO101LeaderConfig, SO101Leader
# from lerobot.robots.so101_follower import SO101FollowerConfig, SO101Follower

# record utils
from lerobot.utils.control_utils import init_keyboard_listener
from lerobot.utils.utils import log_say
from lerobot.utils.visualization_utils import _init_rerun, log_rerun_data
from lerobot.record import record_loop

# env
from lerobot.envs.utils import env_to_policy_features

# torch
from torch import cuda
from torchvision.transforms.functional import to_pil_image

# utils
from dotenv import load_dotenv
from pprint import pprint
import numpy as np
import time

# my code
from src.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR, EVAL_DIR, OUTPUTS_DIR

# my env
from envs.so101_env_config import SO101EnvConfig, make_so101_env
from src.robot_utils import JointSpaceNormalizer, synthetic_leader_dataset

# set up os_env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

Using device: cuda
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Left arrow key pressed. Exiting loop and rerecord the last episode...Left arrow key pressed. Exiting loop and rerecord the last episode...

Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Right arrow key pressed. Exiting loop...Right arrow key pressed. Exiting loop...

Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...Right arrow key pressed. Exiting loop...

Right arrow key pressed. Exiti

Configuration:

In [116]:
PUSH_TO_HUB = False

### Build Env

In [122]:
env_cfg = SO101EnvConfig(
    task     = "TableLegAssembleTask",
    obs_type = "pixels_agent_pos",
    device = device
    )
pprint(env_cfg)
env = make_so101_env(env_cfg, torch_actions = False)

SO101EnvConfig(task='TableLegAssembleTask',
               fps=30,
               features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>,
                                                 shape=(6,)),
                         'agent_pos': PolicyFeature(type=<FeatureType.STATE: 'STATE'>,
                                                    shape=(6,)),
                         'pixels/top_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                                         shape=(480, 640, 3)),
                         'pixels/wrist_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                                           shape=(480, 640, 3))},
               features_map={'action': 'action',
                             'agent_pos': 'observation.state',
                             'env_state': 'observation.environment_state',
                             'pixels/top_cam': 'observation.images.top_cam',
                  

### Connect Leader

In [111]:
teleop_config = SO101LeaderConfig(
    port="COM8",
    id="so_101_leader",
    calibration_dir = CALIBS_DIR
)

teleop = SO101Leader(teleop_config)

# make joint space transofrmation class
js_normalizer = JointSpaceNormalizer(teleop, env.unwrapped.get_joint_range())

In [102]:
try:
    teleop.connect()
    print("Teleop device connected successfully.")
except Exception as e:
    print(e)


Could not connect on port 'COM8'. Make sure you are using the correct port.
Try running `python -m lerobot.find_port`



In [ ]:
# # from lerobot.envs.utils import env_to_policy_features
# # env_to_policy_features(env_cfg)
# TODO handle datasets
# dataset.utils.build_dataset_frame

### Teleop Loop

In [126]:
synthetic_actions = synthetic_leader_dataset(teleop, n_steps=500)

In [ ]:
# init rr + kb listener
listener, events = init_keyboard_listener()
_init_rerun(session_name="teleop_env")

done = False
frame_time = 1.0 / env_cfg.fps
step_idx = 0

while not done and step_idx < len(synthetic_actions):
    loop_start = time.time()
    
    # Check keyboard flags
    if events["exit_early"]:
        pass
    if events["rerecord_episode"]:
        pass
    if events["stop_recording"]:
        print("Stopping data recording.")
        break
    
    # 1. Get leader action, map to env space
    # leader_action = teleop.get_action() TODO temporary for testing
    leader_action = synthetic_actions[step_idx]
    leader_action_env = js_normalizer.robot_to_mujoco(leader_action)
    step_idx += 1
    
    # 2. Step env
    obs, reward, done, truncated, info = env.step(leader_action_env)
    
    # 3. Log to rerun
    obs_to_log = {
        k: (v.detach().cpu().squeeze(0).permute(1, 2, 0).numpy().clip(0, 255) * 255).astype(np.uint8)
        if "images" in k else v.detach().cpu().squeeze(0).float().numpy()
        for k, v in obs.items()
    }
    action_to_log = {"action": np.array(leader_action_env)}
    log_rerun_data(obs_to_log, action_to_log)
    
    # 4. log to dataset
    pass
    
    # 5. Maintain FPS
    elapsed = time.time() - loop_start
    time.sleep(max(0, frame_time - elapsed))

# stop listener when done
if listener:
    listener.stop()

d:\Anaconda3\envs\lerobot2\lib\site-packages\rerun_sdk\rerun\archetypes\image_ext.py:221: RerunWarning: Expected 1, 3, or 4 channels; got 640
  _send_warning_or_raise(f"Expected 1, 3, or 4 channels; got {channels}")
d:\Anaconda3\envs\lerobot2\lib\site-packages\rerun_sdk\rerun\archetypes\image_ext.py:221: RerunWarning: Encountered Error while sending warning
  _send_warning_or_raise(f"Expected 1, 3, or 4 channels; got {channels}")


Right arrow key pressed. Exiting loop...Right arrow key pressed. Exiting loop...

Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Left arrow key pressed. Exiting loop and rerecord the last episode...Left arrow key pressed. Exiting loop and rerecord the last episode...

Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...


In [ ]:
teleop.disconnect()
if PUSH_TO_HUB:
    dataset.push_to_hub()